# Environment Setup

I used this model to test out building and training the model. I used colab's GPUs.

### Packages

In [ ]:
# %pip install transformers h5py pandas pyarrow protobuf

In [ ]:
# %pip install torch

# install cuda on windows
# %pip install torch --index-url https://download.pytorch.org/whl/cu126

### Mount drive for accessing data in Colab

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# Training Model

CUDA device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Constants

In [ ]:
# H5_PATH     = "/content/drive/MyDrive/capstone/data/merged_audio_500.h5" # <--- UPDATE THIS PATH
# META_PATH   = "/content/drive/MyDrive/capstone/data/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
H5_PATH     = "../data_TEMP/merged_audio_500.h5" # <--- UPDATE THIS PATH
META_PATH   = "../data_TEMP/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
BATCH_SIZE = 1      # Reduced batch size for memory optimization. Try 1 first. <--- MODIFIED
SAMPLE_RATE = 16_000
MAX_NUM_TURNS = 5

Dataset + loader

In [ ]:
# Local
from ConvoStyleDataset import ConvoStyleDataset, collate_pad

# Drive
# put here


dataset = ConvoStyleDataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    num_turns=MAX_NUM_TURNS # Changed from 5 to 1 to allow loading single-turn utterances for diagnosis
    # max_len_sec=10.0, # Uncomment and adjust this value (e.g., 10 seconds) for further memory optimization.
    # no max_len_sec -- collate_pad handles variable lengths
)

from torch.utils.data import DataLoader
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,
)

Utterance Encoder

In [ ]:
from transformers import BertTokenizer, BertModel, WavLMModel, Wav2Vec2FeatureExtractor

bert_tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model      = BertModel.from_pretrained("bert-base-uncased").half().to(device)  # using half() to limit gpu memory for quick tests

# WavLM only needs a feature extractor, not a full processor with tokenizer
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model     = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").half().to(device)

for model in (bert_model, wavlm_model):
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("Encoders loaded and frozen")

In [ ]:
# Local
from UtteranceEncoder import DualModalityEmbedder

# Drive


# initialize Embedder for bimodal utterances
embedder = DualModalityEmbedder(
    text_encoder_model_pretrained=bert_model
    , audio_encoder_model_pretrained=wavlm_model
    , tokenizer=bert_tokenizer
    , processor=wavlm_processor
).to(device)

print("Embedder initialized.")


# Testing
# embedder.eval()

# # Re-define batch here as it was not found in the kernel state
# batch = next(iter(loader))

# text_emb, audio_emb = embedder(
#     batch["audio"].to(device).half(),
#     batch["lengths"],
#     batch["transcription"],
#     text_only=batch["text_only"].to(device),
# )

In [ ]:
# Local
from IntraModalDialogueEncoder import ContextAwareTransformer, SpeakerAwareTransformer

# Drive


# Define transformer parameters
d_model = 768 # Embedding dimension from BERT/WavLM
nhead = 8 # Number of attention heads
num_encoder_layers = 2 # Number of transformer encoder layers
dim_feedforward = 2048 # Dimension of the feedforward network model

# Instantiate the ContextAwareTransformer and move it to the device
context_transformer = ContextAwareTransformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(device)

# Testing
# # pass audio and text embeddings separately
# context_aware_embeddings_audio = context_transformer(audio_emb)
# context_aware_embeddings_text = context_transformer(text_emb)

# print("Context-aware audio embeddings shape:", context_aware_embeddings_audio.shape)
# print("Context-aware audio embeddings shape:", context_aware_embeddings_text.shape)
# assert context_aware_embeddings_audio.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
# assert context_aware_embeddings_text.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
# print("Context-aware audio transformer output shape assertion passed.")

# Instantiate the SpeakerAwareTransformer and move it to the device
speaker_aware_transformer = SpeakerAwareTransformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(device)


# TESTING
# # Pass audio embeddings through the speaker-aware transformer
# intra_speaker_aware_audio_emb, inter_speaker_aware_audio_emb = speaker_aware_transformer(
#     audio_emb,
#     speaker_ids_list=batch["speaker_id"]
# )

# # Pass text embeddings through the speaker-aware transformer
# intra_speaker_aware_text_emb, inter_speaker_aware_text_emb = speaker_aware_transformer(
#     text_emb,
#     speaker_ids_list=batch["speaker_id"]
# )

# print("Intra-speaker aware audio embeddings shape:", intra_speaker_aware_audio_emb.shape)
# print("Inter-speaker aware audio embeddings shape:", inter_speaker_aware_audio_emb.shape)
# print("Intra-speaker aware text embeddings shape:", intra_speaker_aware_text_emb.shape)
# print("Inter-speaker aware text embeddings shape:", inter_speaker_aware_text_emb.shape)

# assert intra_speaker_aware_audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
# assert inter_speaker_aware_audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
# assert intra_speaker_aware_text_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
# assert inter_speaker_aware_text_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
# print("Speaker-aware transformer output shapes assertion passed.")